In [1]:
import pandas as pd

# Load the dataset
df = pd.read_csv("telco.csv")

# Display dataset information
print("Dataset shape:", df.shape)
print("\nFirst 5 records:")
print(df.head())

print("\nColumn information:")
df.info()

# Check duplicate records
print("\nDuplicate rows:", df.duplicated().sum())

# Clean TotalCharges
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"], errors="coerce"
)

# Check missing values
print("\nMissing values:")
print(df.isnull().sum())

# Analyze customer churn
print("\nChurn distribution:")
print(df["Churn"].value_counts())

print("\nChurn percentage:")
print((df["Churn"].value_counts(normalize=True) * 100).round(2))

Dataset shape: (7043, 21)

First 5 records:
   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0     Yes         No       1           No   
1  5575-GNVDE    Male              0      No         No      34          Yes   
2  3668-QPYBK    Male              0      No         No       2          Yes   
3  7795-CFOCW    Male              0      No         No      45           No   
4  9237-HQITU  Female              0      No         No       2          Yes   

      MultipleLines InternetService OnlineSecurity  ... DeviceProtection  \
0  No phone service             DSL             No  ...               No   
1                No             DSL            Yes  ...              Yes   
2                No             DSL            Yes  ...               No   
3  No phone service             DSL            Yes  ...              Yes   
4                No     Fiber optic             No  ...               No   

  TechSupport Stre

In [2]:
# Calculate churn rate for each contract type
contract_analysis = (
    df.groupby("Contract", observed=True)
      .agg(
          Total_Customers=("customerID", "count"),
          Churned_Customers=(
              "Churn", lambda x: (x == "Yes").sum()
          ),
          Average_Monthly_Charges=("MonthlyCharges", "mean")
      )
      .reset_index()
)

contract_analysis["Churn_Rate_Percent"] = (
    contract_analysis["Churned_Customers"]
    / contract_analysis["Total_Customers"] * 100
).round(2)

print(contract_analysis)

         Contract  Total_Customers  Churned_Customers  \
0  Month-to-month             3875               1655   
1        One year             1473                166   
2        Two year             1695                 48   

   Average_Monthly_Charges  Churn_Rate_Percent  
0                66.398490               42.71  
1                65.048608               11.27  
2                60.770413                2.83  


In [3]:

import pandas as pd

# 1. Load dataset
df = pd.read_csv("telco.csv")

print("Original dataset:", df.shape)

# 2. Remove duplicate records
df = df.drop_duplicates()

# 3. Clean column names and text values
df.columns = df.columns.str.strip()

for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].str.strip()

# 4. Convert TotalCharges to numeric
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"], errors="coerce"
)

# 5. Check missing values
print("\nMissing values before cleaning:")
print(df.isnull().sum())

# These blank TotalCharges values are not usable for
# charge-based analysis. Keep the original tenure records
# for churn analysis, but exclude them from charge analysis.
df_clean = df.dropna(subset=["TotalCharges"]).copy()

# 6. Check data quality
print("\nCleaned dataset:", df_clean.shape)
print("Duplicate rows:", df_clean.duplicated().sum())
print("Missing values remaining:", df_clean.isnull().sum().sum())

# 7. Save cleaned data
df_clean.to_csv("telco_cleaned.csv", index=False)

print("\nCleaned dataset saved successfully!")


Original dataset: (7043, 21)

Missing values before cleaning:
customerID           0
gender               0
SeniorCitizen        0
Partner              0
Dependents           0
tenure               0
PhoneService         0
MultipleLines        0
InternetService      0
OnlineSecurity       0
OnlineBackup         0
DeviceProtection     0
TechSupport          0
StreamingTV          0
StreamingMovies      0
Contract             0
PaperlessBilling     0
PaymentMethod        0
MonthlyCharges       0
TotalCharges        11
Churn                0
dtype: int64

Cleaned dataset: (7032, 21)
Duplicate rows: 0
Missing values remaining: 0

Cleaned dataset saved successfully!


In [4]:

%pip install pyspark


In [5]:

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# Start Spark
spark = (
    SparkSession.builder
    .appName("TelecomChurnAnalytics")
    .master("local[*]")
    .getOrCreate()
)

# Load the original CSV into Spark
spark_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("telco.csv")
)

print("Total records:", spark_df.count())
print("Total columns:", len(spark_df.columns))

spark_df.printSchema()
spark_df.show(5, truncate=False)


Total records: 7043
Total columns: 21
root
 |-- customerID: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- SeniorCitizen: integer (nullable = true)
 |-- Partner: string (nullable = true)
 |-- Dependents: string (nullable = true)
 |-- tenure: integer (nullable = true)
 |-- PhoneService: string (nullable = true)
 |-- MultipleLines: string (nullable = true)
 |-- InternetService: string (nullable = true)
 |-- OnlineSecurity: string (nullable = true)
 |-- OnlineBackup: string (nullable = true)
 |-- DeviceProtection: string (nullable = true)
 |-- TechSupport: string (nullable = true)
 |-- StreamingTV: string (nullable = true)
 |-- StreamingMovies: string (nullable = true)
 |-- Contract: string (nullable = true)
 |-- PaperlessBilling: string (nullable = true)
 |-- PaymentMethod: string (nullable = true)
 |-- MonthlyCharges: double (nullable = true)
 |-- TotalCharges: string (nullable = true)
 |-- Churn: string (nullable = true)

+----------+------+-------------+-------+--

In [6]:

# Calculate churn by contract type
contract_churn = (
    spark_df
    .groupBy("Contract")
    .agg(
        F.count("*").alias("Total_Customers"),
        F.sum(
            F.when(F.col("Churn") == "Yes", 1).otherwise(0)
        ).alias("Churned_Customers")
    )
    .withColumn(
        "Churn_Rate_Percent",
        F.round(
            F.col("Churned_Customers") /
            F.col("Total_Customers") * 100,
            2
        )
    )
    .orderBy(F.desc("Churn_Rate_Percent"))
)

contract_churn.show(truncate=False)


+--------------+---------------+-----------------+------------------+
|Contract      |Total_Customers|Churned_Customers|Churn_Rate_Percent|
+--------------+---------------+-----------------+------------------+
|Month-to-month|3875           |1655             |42.71             |
|One year      |1473           |166              |11.27             |
|Two year      |1695           |48               |2.83              |
+--------------+---------------+-----------------+------------------+



In [7]:

from pyspark.sql import functions as F

# Convert charge columns to numeric safely
df_spark = (
    spark_df
    .withColumn("MonthlyCharges", F.col("MonthlyCharges").cast("double"))
    .withColumn("TotalCharges", F.col("TotalCharges").cast("double"))
)

# Overall customer KPIs
kpis = df_spark.agg(
    F.count("*").alias("Total_Customers"),
    F.sum(
        F.when(F.col("Churn") == "Yes", 1).otherwise(0)
    ).alias("Churned_Customers"),
    F.sum(
        F.when(F.col("Churn") == "No", 1).otherwise(0)
    ).alias("Retained_Customers"),
    F.round(F.avg("MonthlyCharges"), 2).alias("Avg_Monthly_Charges")
)

kpis.show()

# Calculate overall churn rate
churn_rate = (
    df_spark
    .agg(
        F.round(
            F.avg(
                F.when(F.col("Churn") == "Yes", 1.0)
                 .otherwise(0.0)
            ) * 100,
            2
        ).alias("Churn_Rate_Percent")
    )
)

churn_rate.show()


+---------------+-----------------+------------------+-------------------+
|Total_Customers|Churned_Customers|Retained_Customers|Avg_Monthly_Charges|
+---------------+-----------------+------------------+-------------------+
|           7043|             1869|              5174|              64.76|
+---------------+-----------------+------------------+-------------------+

+------------------+
|Churn_Rate_Percent|
+------------------+
|             26.54|
+------------------+



In [8]:

# Analyze churn by different customer segments
segments = [
    "Contract",
    "InternetService",
    "PaymentMethod",
    "SeniorCitizen"
]

for segment in segments:
    print(f"\n--- Churn Analysis by {segment} ---")

    analysis = (
        df_spark
        .groupBy(segment)
        .agg(
            F.count("*").alias("Total_Customers"),
            F.sum(
                F.when(F.col("Churn") == "Yes", 1).otherwise(0)
            ).alias("Churned_Customers")
        )
        .withColumn(
            "Churn_Rate_Percent",
            F.round(
                F.col("Churned_Customers") /
                F.col("Total_Customers") * 100, 2
            )
        )
        .orderBy(F.desc("Churn_Rate_Percent"))
    )

    analysis.show(truncate=False)



--- Churn Analysis by Contract ---
+--------------+---------------+-----------------+------------------+
|Contract      |Total_Customers|Churned_Customers|Churn_Rate_Percent|
+--------------+---------------+-----------------+------------------+
|Month-to-month|3875           |1655             |42.71             |
|One year      |1473           |166              |11.27             |
|Two year      |1695           |48               |2.83              |
+--------------+---------------+-----------------+------------------+


--- Churn Analysis by InternetService ---
+---------------+---------------+-----------------+------------------+
|InternetService|Total_Customers|Churned_Customers|Churn_Rate_Percent|
+---------------+---------------+-----------------+------------------+
|Fiber optic    |3096           |1297             |41.89             |
|DSL            |2421           |459              |18.96             |
|No             |1526           |113              |7.4               |
+---